# Project Lab 2: Recommender, Credit Scorecard and Causal Analysis

Notebook for **Chapter 16** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch16.html

Tip: change a number, predict the output, then run the cell again.

## 16.1 Project: Two-stage recommender with NDCG@10 and cold start (#27)

**Goal and why employers care.** Real recommenders cannot score every item for every user in real time. They use two stages: a fast *candidate generator* picks about 100 items, then a slower *ranker* orders them. Building this, measuring it with a ranking metric and handling brand-new items shows you understand how large systems work.

In [ ]:
import numpy as np
# Step 1: synthetic MovieLens-style data (users, items, ratings, cold items)
r = np.random.default_rng(11)
U, I, K = 600, 300, 6
uf = r.normal(size=(U,K)); itf = r.normal(size=(I,K))
pop = r.normal(size=I)*0.8
score = uf@itf.T/np.sqrt(K) + pop
# user rates 40 items drawn with popularity bias; rating = 1..5 from score
cold = set(range(I-30, I))                          # last 30 items are cold: no training interactions
rows=[]
for u in range(U):
    pu = np.exp(1.2*score[u]); pu/=pu.sum()
    items = r.choice(I, 40, replace=False, p=pu)
    for i in items:
        rating = int(np.clip(np.round(3+score[u,i]+r.normal(0,.5)),1,5))
        rows.append((u,i,rating))
rows = np.array(rows)
# hold out each user's 5 highest-rated items (relevant = rating>=4) as test; cold items only in test
test=[]; train=[]
for u in range(U):
    ru = rows[rows[:,0]==u]
    rel = ru[(ru[:,2]>=4)]
    t_items = set(rel[:5,1]) if len(rel) else set()
    for (_,i,rt) in ru:
        (test if (i in t_items or i in cold) else train).append((u,i,rt))
test=np.array([x for x in test if x[2]>=4]); train=np.array(train)
print("train", len(train), "test positives", len(test), "cold-item test positives", int(np.isin(test[:,1], list(cold)).sum()))
seen = {u:set(train[train[:,0]==u][:,1]) for u in range(U)}
rel_by_user = {u:set(test[test[:,0]==u][:,1]) for u in range(U)}

# Step 2: metric and popularity baseline
def ndcg_at_k(ranked, relevant, k=10):
    dcg = sum(1/np.log2(i+2) for i,x in enumerate(ranked[:k]) if x in relevant)
    ideal = sum(1/np.log2(i+2) for i in range(min(k,len(relevant))))
    return dcg/ideal if ideal else np.nan
def evaluate(rank_fn, k=10):
    v=[ndcg_at_k(rank_fn(u), rel_by_user[u], k) for u in range(U) if rel_by_user[u]]
    return float(np.nanmean(v))

# baseline: popularity
cnt = np.bincount(train[:,1], minlength=I)
poprank = np.argsort(-cnt)
pop_fn = lambda u: [i for i in poprank if i not in seen[u]]
print("popularity NDCG@10:", round(evaluate(pop_fn),3))

# stage 1: candidate generation = item-item cosine on 'liked' (>=4) matrix, 100 candidates
M = np.zeros((U,I)); M[train[train[:,2]>=4][:,0], train[train[:,2]>=4][:,1]] = 1
norm = np.linalg.norm(M,axis=0)+1e-9
S = (M.T@M)/np.outer(norm,norm)
# Step 3: stage 1, candidate generation (item-item + popularity)
feat = itf + r.normal(0,.5,itf.shape)                   # item content features (noisy)
def candidates(u, n_cf=70, n_pop=30):
    liked = list(np.where(M[u]>0)[0])
    s_ = S[:, liked].sum(axis=1) if liked else np.zeros(I)
    s_ = s_.copy(); s_[list(seen[u])] = -1
    cf = list(np.argsort(-s_)[:n_cf])
    pp = [i for i in poprank if i not in seen[u] and i not in cf][:n_pop]
    return cf + pp
cg_fn = lambda u: candidates(u)
print("candidate gen only NDCG@10:", round(evaluate(cg_fn),3))
recall = np.mean([len(set(candidates(u)) & rel_by_user[u])/len(rel_by_user[u]) for u in range(U) if rel_by_user[u]])
print("candidate recall@100 (item-item + popularity):", round(float(recall),3))

# Step 4: stage 2, ranker (biases + ALS factors): ranker = matrix factorisation by ALS-lite (SVD of mean-centred ratings) blended with popularity
R = np.zeros((U,I)); Mk = np.zeros((U,I))
R[train[:,0],train[:,1]] = train[:,2]; Mk[train[:,0],train[:,1]] = 1
mu = R.sum()/Mk.sum()
bi = ((R-mu)*Mk).sum(0)/(Mk.sum(0)+5)                  # item bias, shrunk
bu = (((R-mu)-bi)*Mk).sum(1)/(Mk.sum(1)+5)             # user bias
def als(Rc, Mk, k=6, lam=8.0, iters=12, seed=0):
    g = np.random.default_rng(seed); P = g.normal(0,.1,(Rc.shape[0],k)); Q = g.normal(0,.1,(Rc.shape[1],k))
    for _ in range(iters):
        for u in range(P.shape[0]):
            m = Mk[u]>0; A = Q[m]; P[u] = np.linalg.solve(A.T@A+lam*np.eye(k), A.T@Rc[u,m])
        for i in range(Q.shape[0]):
            m = Mk[:,i]>0
            if m.any(): A = P[m]; Q[i] = np.linalg.solve(A.T@A+lam*np.eye(k), A.T@Rc[m,i])
    return P, Q
Pf, Qf = als(R-mu-bi-bu[:,None], Mk)
pred = mu + bi[None,:] + bu[:,None] + Pf@Qf.T
# Step 5: cold items: feature -> predicted-score columns learn feature -> predicted-score columns on warm items, apply to cold items
from sklearn.linear_model import Ridge
warm = [i for i in range(I) if i not in cold]; cl = sorted(cold)
W = Ridge(alpha=5).fit(feat[warm], pred[:, warm].T)
pred_full = pred.copy(); pred_full[:, cl] = W.predict(feat[cl]).T
# add a small cold-item content source to candidates
def cands2(u):
    c = candidates(u, 60, 20)
    liked = [i for i in seen[u] if R[u,i]>=4]
    w = feat[liked].mean(axis=0) if liked else np.zeros(K)
    extra = [i for i in np.argsort(-(feat[cl]@w)) ]
    return c + [cl[j] for j in extra[:20]]
def rank_fn(u):
    return sorted(cands2(u), key=lambda i: -pred_full[u,i])
print("two-stage NDCG@10:", round(evaluate(rank_fn),3))
cold_rel = {u: rel_by_user[u] & cold for u in range(U)}
def cold_eval(fn):
    v=[ndcg_at_k(fn(u), cold_rel[u]) for u in range(U) if cold_rel[u]]; return float(np.mean(v)), len(v)
# Step 6: fair view: judge only on warm positives (cold items cannot be found from history)
warm_rel = {u: rel_by_user[u]-cold for u in range(U)}
def warm_eval(fn):
    return float(np.mean([ndcg_at_k(fn(u), warm_rel[u]) for u in range(U) if warm_rel[u]]))
print("warm-only NDCG@10 popularity:", round(warm_eval(pop_fn),3))
print("warm-only NDCG@10 two-stage :", round(warm_eval(rank_fn),3))
# Step 7: cold-start check: rank only the 30 new items for each user who liked some of them
g = np.random.default_rng(0)
rand_fn = lambda u: list(g.permutation(cl))
feat_fn = lambda u: sorted(cl, key=lambda i: -pred_full[u,i])
print("cold-catalogue NDCG@10 random      :", tuple(round(x,3) for x in cold_eval(rand_fn)))
print("cold-catalogue NDCG@10 feature->emb:", tuple(round(x,3) for x in cold_eval(feat_fn)))

## 16.2 Project: Credit-risk scorecard with WoE, KS, Gini and PSI (#28)

**Goal and why employers care.** Banks still rely on scorecards: a points table that anyone can audit. Building one proves you can bin variables safely, fit a logistic model, measure separation (KS, Gini), convert to points and monitor drift (PSI). It is a classic of regulated data science.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

def make_loans(n=30000, seed=5, shift=0.0):
    r = np.random.default_rng(seed)
    age = np.clip(r.normal(40+shift*8,11,n),21,70).round()
    income = r.lognormal(10.6-0.15*shift,0.5,n).round(-2)
    util = np.clip(r.beta(2,3,n)+0.1*shift,0,1.2).round(2)
    late = r.poisson(0.5+0.4*shift,n)
    z = -3.2 - 0.025*(age-40) - 0.6*np.log(income/40000) + 2.0*util + 0.55*late
    bad = (r.random(n) < 1/(1+np.exp(-z))).astype(int)
    return pd.DataFrame(dict(age=age,income=income,util=util,late=late,bad=bad))
# Step 1: data and split
df = make_loans(); print("bad rate", round(df.bad.mean(),3))
train = df.iloc[:20000].copy(); test = df.iloc[20000:].copy()

# Step 2: WoE / IV
def woe_table(x, y, edges):
    b = pd.cut(x, edges, include_lowest=True)
    g = pd.crosstab(b, y).rename(columns={0:"good",1:"bad"}) + 0.5     # smoothing avoids log(0)
    g.columns.name=None
    g["dist_good"]=g.good/g.good.sum(); g["dist_bad"]=g.bad/g.bad.sum()
    g["woe"]=np.log(g.dist_good/g.dist_bad)
    g["iv"]=(g.dist_good-g.dist_bad)*g.woe
    return g
def quant_edges(x, k=5): return sorted(set(np.quantile(x, np.linspace(0,1,k+1))))
edges = {c: quant_edges(train[c]) for c in ["age","income","util"]}
edges["late"] = [-0.5,0.5,1.5,2.5,99]
ivs = {}
for c,e in edges.items():
    t = woe_table(train[c], train.bad, e); ivs[c] = t.iv.sum()
print("IV:", {k: round(float(v),3) for k,v in ivs.items()})
print(woe_table(train["util"], train.bad, edges["util"]).round(3)[["good","bad","woe","iv"]])

# Step 3: model on WoE features
def to_woe(d):
    out = pd.DataFrame(index=d.index)
    for c,e in edges.items():
        t = woe_table(train[c], train.bad, e)["woe"]
        b = pd.cut(d[c].clip(e[0], e[-1]), e, include_lowest=True)
        out[c] = b.map(t).astype(float)
    return out
Xtr, Xte = to_woe(train), to_woe(test)
m = LogisticRegression(C=1.0, max_iter=1000).fit(Xtr, train.bad)
print("coefs:", dict(zip(Xtr.columns, [round(float(c),2) for c in m.coef_[0]])))
ptr, pte = m.predict_proba(Xtr)[:,1], m.predict_proba(Xte)[:,1]
def ks(y,p): return ks_2samp(p[y==1], p[y==0]).statistic
for nm,y,p in [("train",train.bad,ptr),("test",test.bad,pte)]:
    auc=roc_auc_score(y,p); print(f"{nm}: AUC={auc:.3f} Gini={2*auc-1:.3f} KS={ks(y.values,p):.3f}")

# Step 4: points; Step 5: PSI: score = 600 + 20/ln2*ln(odds_good) , PDO=20, base odds 50:1 at 600
pdo, base, odds0 = 20, 600, 50
factor = pdo/np.log(2); offset = base - factor*np.log(odds0)
def score(p_bad): return offset + factor*np.log((1-p_bad)/p_bad)
s_tr, s_te = score(ptr), score(pte)
print("score range test:", int(s_te.min()), int(s_te.max()), "mean", round(s_te.mean(),1))

def psi(expected, actual, edges):
    e = np.histogram(expected, edges)[0]/len(expected); a = np.histogram(actual, edges)[0]/len(actual)
    e, a = np.clip(e,1e-4,None), np.clip(a,1e-4,None)
    return float(np.sum((a-e)*np.log(a/e)))
sedges = np.quantile(s_tr, np.linspace(0,1,11)); sedges[0]-=1; sedges[-1]+=1
print("PSI train vs test:", round(psi(s_tr, s_te, sedges),4))
drift = make_loans(10000, seed=9, shift=1.0)
s_dr = score(m.predict_proba(to_woe(drift))[:,1])
print("PSI train vs drifted applicants:", round(psi(s_tr, s_dr, sedges),3))

## 16.3 Project: Causal analysis with propensity scores and difference-in-differences (#30)

**Goal and why employers care.** Many business questions ("did the loyalty club raise spending?") cannot be tested with a coin flip. Employers want to see that you know the difference between correlation and effect, that you can use two standard tools, and, most of all, that you test your own answer for weak spots.

In [ ]:
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression, LinearRegression

# Step 1: data: propensity scores. Treatment = joined loyalty club; outcome = spend next 3 months. True effect = +10
r = np.random.default_rng(21); n = 20000
engage = r.normal(size=n); age = r.normal(40,10,n); prior = 50 + 15*engage + r.normal(0,10,n)
p_join = 1/(1+np.exp(-(-1.0 + 1.2*engage + 0.02*(age-40))))
T = (r.random(n) < p_join).astype(int)
Y = 60 + 12*engage + 0.3*(age-40) + 10*T + r.normal(0,15,n)
df = pd.DataFrame(dict(engage=engage, age=age, prior=prior, T=T, Y=Y))
print("naive diff:", round(df.Y[df['T']==1].mean()-df.Y[df['T']==0].mean(),2), "(true effect 10)")

# Step 2: propensity score and balance observe: prior spend (proxy of engagement) and age
Xc = df[["prior","age"]]
Xs = (Xc-Xc.mean())/Xc.std()
ps = LogisticRegression().fit(Xs, df['T']).predict_proba(Xs)[:,1]
df["ps"]=ps
print("propensity range treated:", df.ps[df['T']==1].quantile([.01,.99]).round(2).tolist(), "control:", df.ps[df['T']==0].quantile([.01,.99]).round(2).tolist())
def smd(x, t):
    a,b = x[t==1], x[t==0]; return (a.mean()-b.mean())/np.sqrt((a.var()+b.var())/2)
print("SMD before:", {c: round(float(smd(df[c], df['T'])),2) for c in ["prior","age"]})
# Step 3: IPW with trimming
d = df[(df.ps>0.02)&(df.ps<0.98)]
w = np.where(d['T']==1, 1/d.ps, 1/(1-d.ps))
ate = np.average(d.Y[d['T']==1], weights=w[d['T']==1]) - np.average(d.Y[d['T']==0], weights=w[d['T']==0])
print("IPW ATE:", round(ate,2))
def wsmd(x,t,w):
    m1=np.average(x[t==1],weights=w[t==1]); m0=np.average(x[t==0],weights=w[t==0])
    return (m1-m0)/np.sqrt((x[t==1].var()+x[t==0].var())/2)
print("SMD after weighting:", {c: round(float(wsmd(d[c].values, d['T'].values, w)),2) for c in ["prior","age"]})
# bootstrap CI
bs=[]
g=np.random.default_rng(0)
for _ in range(200):
    s=d.sample(len(d), replace=True, random_state=int(g.integers(1e9)))
    ww=np.where(s['T']==1,1/s.ps,1/(1-s.ps))
    bs.append(np.average(s.Y[s['T']==1],weights=ww[s['T']==1])-np.average(s.Y[s['T']==0],weights=ww[s['T']==0]))
print("bootstrap 95% CI:", np.round(np.percentile(bs,[2.5,97.5]),2).tolist())
# Step 4: sensitivity the proxy (hidden confounder) -> estimate with age only
Xa = ((df[["age"]]-df.age.mean())/df.age.std())
pa = LogisticRegression().fit(Xa, df['T']).predict_proba(Xa)[:,1]
wa = np.where(df['T']==1,1/pa,1/(1-pa))
ate_h = np.average(df.Y[df['T']==1],weights=wa[df['T']==1]) - np.average(df.Y[df['T']==0],weights=wa[df['T']==0])
print("IPW with engagement hidden (age only):", round(ate_h,2))

# Step 5: DiD data: difference-in-differences. 20 cities, 10 get free delivery from month 7. true effect = +8 orders/1000 customers
r = np.random.default_rng(4)
rows=[]
for c in range(20):
    treated = int(c<10); base = 100 + 5*r.normal() + (6 if treated else 0)   # treated cities start higher (levels differ: fine)
    for m in range(12):
        post = int(m>=6)
        y = base + 1.5*m + 8*treated*post + r.normal(0,2)
        rows.append((c,m,treated,post,y))
p = pd.DataFrame(rows, columns=["city","month","treated","post","y"])
means = p.groupby(["treated","post"]).y.mean().unstack()
print(means.round(2))
did = (means.loc[1,1]-means.loc[1,0]) - (means.loc[0,1]-means.loc[0,0])
print("DiD estimate:", round(did,2))
# Step 6: uncertainty with city-clustered bootstrap
def did_est(q):
    mm = q.groupby(["treated","post"]).y.mean().unstack(); return (mm.loc[1,1]-mm.loc[1,0])-(mm.loc[0,1]-mm.loc[0,0])
g=np.random.default_rng(1); est=[]
for _ in range(500):
    cs_t = g.choice(range(10),10); cs_c = g.choice(range(10,20),10)
    q = pd.concat([p[p.city==c] for c in list(cs_t)+list(cs_c)])
    est.append(did_est(q))
print("cluster bootstrap 95% CI:", np.round(np.percentile(est,[2.5,97.5]),2).tolist())
# Step 7: placebo and pre-trend pretend treatment began at month 3, use only pre months 0-5
pre = p[p.month<6].copy(); pre["post"]=(pre.month>=3).astype(int)
print("placebo DiD (fake start month 3, pre-data only):", round(did_est(pre),2))
# parallel-trend check: slope of pre-period by group
for tr in [1,0]:
    q = p[(p.treated==tr)&(p.post==0)].groupby("month").y.mean()
    print("pre-trend slope treated=%d:"%tr, round(np.polyfit(q.index,q.values,1)[0],2))

## 16.4 Worked Debugging Problems and Marking Guide

Each project in this chapter has a classic failure that produces a flattering number. Here are three, with a test that catches each. Spot the bug first, then read the steps.

In [ ]:
import numpy as np
def dcg(rels): return sum(r / np.log2(i + 2) for i, r in enumerate(rels))
shown = [0, 1, 0, 1, 0]            # relevance of the 5 items we ranked
total_relevant = 3                 # the user actually has 3 relevant items in the catalogue
wrong_ideal = dcg(sorted(shown, reverse=True))
right_ideal = dcg([1] * min(total_relevant, 5))
print("DCG@5:", round(dcg(shown), 4))
print("wrong ideal (sorts the shown list):", round(wrong_ideal, 4), " NDCG", round(dcg(shown) / wrong_ideal, 3))
print("right ideal (all relevant items):  ", round(right_ideal, 4), " NDCG", round(dcg(shown) / right_ideal, 3))

DCG@5: 1.0616
wrong ideal (sorts the shown list): 1.6309  NDCG 0.651
right ideal (all relevant items):   2.1309  NDCG 0.498


In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score
rng = np.random.default_rng(15)
n = 1000
x = rng.normal(size=n)                                   # a pure-noise feature
y = (rng.random(n) < 0.2).astype(int)                    # labels unrelated to x
idx = rng.permutation(n); tr, te = idx[:600], idx[600:]
def woe_map(x_fit, y_fit, bins=60):
    edges = np.quantile(x_fit, np.linspace(0, 1, bins + 1)); edges[0], edges[-1] = -np.inf, np.inf
    b = np.digitize(x_fit, edges[1:-1])
    g = np.bincount(b, weights=1 - y_fit, minlength=bins) + 0.5
    bd = np.bincount(b, weights=y_fit, minlength=bins) + 0.5
    return edges, np.log((g / g.sum()) / (bd / bd.sum()))
def apply(x_new, edges, woe): return woe[np.digitize(x_new, edges[1:-1])]
# bug: WOE learned from ALL rows, then evaluated on the test rows
e_all, w_all = woe_map(x, y)
leaky = roc_auc_score(y[te], -apply(x[te], e_all, w_all))
e_tr, w_tr = woe_map(x[tr], y[tr])
honest = roc_auc_score(y[te], -apply(x[te], e_tr, w_tr))
print("test AUC with WOE fitted on all rows (leak):", round(leaky, 3), " Gini", round(2*leaky - 1, 3))
print("test AUC with WOE fitted on train only:     ", round(honest, 3), " Gini", round(2*honest - 1, 3))

test AUC with WOE fitted on all rows (leak): 0.718  Gini 0.435
test AUC with WOE fitted on train only:      0.542  Gini 0.083


In [ ]:
import numpy as np
rng = np.random.default_rng(13)
T, T0 = 16, 10
t = np.arange(T)
ctrl = 100 + 1.0*t + rng.normal(0, 0.5, T)
treat = 110 + 2.0*t + rng.normal(0, 0.5, T)                  # already growing faster BEFORE launch
treat[T0:] += 4.0                                            # true effect 4
pre_gap = (treat - ctrl)[:T0]
slope = np.polyfit(t[:T0], pre_gap, 1)[0]
did = (treat[T0:].mean() - treat[:T0].mean()) - (ctrl[T0:].mean() - ctrl[:T0].mean())
print("pre-period gap slope per period:", round(slope, 3), "(should be about 0)")
print("naive DiD estimate:", round(did, 2), " true effect: 4.0")
# fix: allow a group-specific linear trend estimated on the pre-period
cf = np.polyval(np.polyfit(t[:T0], treat[:T0] - ctrl[:T0], 1), t[T0:])
est = ((treat - ctrl)[T0:] - cf).mean()
print("trend-adjusted estimate:", round(est, 2))

pre-period gap slope per period: 1.091 (should be about 0)
naive DiD estimate: 11.83  true effect: 4.0
trend-adjusted estimate: 3.1


## Chapter recap

## Practice set

**1.** (easy) A relevant item appears at ranks 1 and 3 of the top 10; the user has 2 relevant items in total. Compute NDCG@10. **Answer:** `DCG = 1/log2(2) + 1/log2(4) = 1 + 0.5 = 1.5`. Ideal `= 1 + 1/log2(3) = 1.631`. NDCG `= 1.5/1.631 = 0.920`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch16.html